# Expert Knowledge worker:

<p>Working on a question answering agent, to be used by the employees of InsureLLM, an Insurance Tech Company.</p>

<h3>Today's task is split in 3 part:</h3>

Part A: We will divide our documents into CHUNKS. <br>
Part B: We will encode our CHUNKS into VECTORS and put in Chroma. <br>
Part C: We will visualize the vectors. <br>

## Part A: Divide the documents into chunks:

In [ ]:
import os
import glob
from dotenv import load_dotenv
import numpy as np
from langchain_openai import OpenAIEmbeddings                                                                           # Langchain framework for loading OpenAI embeddings
from langchain_chroma import Chroma                                                                                     # Vector database
from langchain_huggingface import HuggingFaceEmbeddings                                                                 # Langchain framework for loading HuggingFace embeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader                                            # Langchain framework for loading documents and text files
from langchain_text_splitters import RecursiveCharacterTextSplitter                                                     # Langchain framework for splitting text into smaller chunks
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import tiktoken

In [ ]:
MODEL = "gpt-5-mini"
db_name = "vector_db"
load_dotenv(override=True)  # Load environment variables from .env file
opeanai_api_key = os.getenv("OPENAI_API_KEY")

### Loading all the documents onto a single source:

In [ ]:
knowledge_base_path = "knowledge-base/**/*.md"

files = glob.glob(knowledge_base_path, recursive=True)
print(f"Total of {len(files)} files found in the knowledge base.")

entire_knowledge_base = ""

for file in files:
    with open(file, "r", encoding = "utf-8") as f:
        entire_knowledge_base += f.read() + '\n\n'

print(f"Total of {len(entire_knowledge_base):,} characters in the entire knowldge base.")

### Checking how much tokens will be generated: [Not Important!!]

In [ ]:
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Totla token count is {token_count:,} tokens.")

### Generating a LangChain Object "Documents" representing all of the documents passed to langchain.

In [ ]:
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob = "**/*.md", loader_cls = TextLoader, show_progress = True, loader_kwargs = {"encoding": "utf-8"})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"{len(documents)} documents loaded from the knowledge base.")

In [ ]:
documents[60]


### Splitting documents into Chunks using RecursiveCharacterTextSplitter:

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks.")
print(f"First chunk {chunks[0]}")

## Part B: Make vectors and store in Chroma

In [ ]:
# Picking an embedding model

# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection() # Completely delete the existing collection if it exists

vector_store = Chroma.from_documents(documents = chunks, embedding = embeddings, persist_directory = db_name)
print(f"VectorStore created with {vector_store._collection.count()} documnets")

In [ ]:
collections = vector_store._collection
count = collections.count()

sample_embedding = collections.get(limit=1, include=["embeddings"])["embeddings"][0]
dimension = len(sample_embedding)
print(f"There are {count} embeddings/vectors in the vector store, each with a dimension of {dimension}.")

### Part C: Visualizing the vectors:

In [ ]:
result = collections.get(include=["metadatas", "documents","embeddings"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
doc_type = [metadata["doc_type"] for metadata in metadatas]
colors = [['blue','green','red','orange'][['products','employees','contracts','company'].index(t)] for t in doc_type]


In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_type, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

### 3D Visualization:

In [ ]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_type, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()